# Large Language Models (LLMs)

Large Language Models (LLMs) are deep learning models trained on massive amounts of text data to understand and generate human language. They represent the current state of the art in natural language processing and have transformed how we interact with AI.

At their core, LLMs are built on the **transformer architecture** (covered in the Transformers notebook), but they introduce numerous innovations in architecture, training, and deployment that enable capabilities far beyond what earlier models could achieve.

### What Makes a Language Model "Large"?

The term "large" refers to both the **number of parameters** and the **scale of training data**. Parameters are the learnable weights inside the neural network. They are the values that get adjusted during training and collectively encode everything the model "knows." More parameters means the model has greater capacity to store patterns, facts, and relationships learned from data.

| Model | Year | Parameters | Training Data |
|-------|------|-----------|---------------|
| GPT-2 | 2019 | 1.5B | 40 GB text |
| GPT-3 | 2020 | 175B | 570 GB text |
| LLaMA 2 | 2023 | 7B–70B | 2 trillion tokens |
| GPT-4 | 2023 | ~1.8T (rumored) | ~13 trillion tokens |
| Llama 3 | 2024 | 8B–405B | 15 trillion tokens |
| DeepSeek-V3 | 2024 | 671B (37B active) | 14.8 trillion tokens |

To put these numbers in perspective: GPT-2 (1.5 billion parameters) was considered impressively large in 2019. Just four years later, models had grown by **1,000×** in parameter count and were being trained on datasets hundreds of times larger. This explosive growth was driven by the discovery that simply making models bigger and feeding them more data consistently improves performance.

As models scale up, they develop **emergent abilities**; capabilities that appear to arise at certain scales rather than improving gradually. These include in-context learning (learning from examples in the prompt without any weight updates), chain-of-thought reasoning (solving problems step by step), and the ability to follow complex multi-step instructions.

### The Modern LLM Landscape

Today's LLM ecosystem includes both **proprietary** and **open-weight** models:

- **Proprietary:** GPT-4/4o (OpenAI), Claude 3.5/4 (Anthropic), Gemini 2 (Google) — accessible only via APIs, with architectures and training details kept confidential.
- **Open-Weight:** Llama 3/4 (Meta), Mistral/Mixtral (Mistral AI), DeepSeek-V3/R1 (DeepSeek), Qwen 2.5 (Alibaba) — model weights are publicly released, allowing anyone to download, run, fine-tune, and study them.

The distinction between "open-weight" and "open-source" is important: most open-weight models release the trained weights and inference code, but not the full training data or training infrastructure. Still, open-weight models have democratized access to frontier AI capabilities and enabled a thriving research ecosystem.

This notebook covers the key architectural innovations, training techniques, and practical tools that define modern LLMs.

---
# Tokenization

Before any text reaches an LLM, it must be converted into **tokens** or the numerical representations that the model can process. Tokenization is the critical first step in the LLM pipeline, and the choice of tokenizer significantly affects model performance, cost, and capability.

Think of tokenization as translating human-readable text into a language the model understands. Just as we break sentences into words when reading, a tokenizer breaks text into smaller pieces that map to entries in a fixed vocabulary.

### Why Not Just Use Words or Characters?

There are several intuitive approaches to breaking text into pieces, but each has significant drawbacks:

- **Word-level tokenization** creates a separate token for every unique word. This struggles with rare words, typos, and morphological variations. A model trained on "running" might not understand "runner" or "runs" because they are treated as completely separate tokens with no shared representation. The vocabulary would also need to be enormous to cover every possible word in every language, including technical jargon, proper nouns, and misspellings.

- **Character-level tokenization** breaks text into individual letters. While this eliminates the unknown-word problem (any text can be represented), it makes sequences extremely long. The word "transformer" becomes 11 tokens instead of 1, meaning the model needs to learn to compose characters into meaningful units — a much harder task. It also means the model's context window holds far less actual content.

Modern LLMs use **subword tokenization**, which strikes an effective balance: common words (like "the", "is", "hello") stay as single tokens, while rare or complex words are broken into meaningful pieces ("unforgettable" → "un" + "forget" + "table"). This keeps the vocabulary manageable (typically 32K–100K entries) while ensuring any text can be represented.

### Byte Pair Encoding (BPE)

**BPE** is the most widely used tokenization algorithm in modern LLMs. It was originally a data compression technique from the 1990s, adapted for NLP by Sennrich et al. in 2016.

#### How BPE Works:

The algorithm builds its vocabulary through an iterative merging process:

1. **Initialize:** Start with a vocabulary of all individual characters (or bytes) in the training data.
2. **Count pairs:** Look at all adjacent token pairs in the corpus and count how often each pair appears.
3. **Merge the most frequent pair:** Take the most common adjacent pair and combine them into a new single token. Add this new token to the vocabulary.
4. **Repeat:** Go back to step 2 and continue merging until the vocabulary reaches the desired size (typically 32K–100K tokens).

**Example — building a vocabulary from the word "lower":**
```
Starting: ['l', 'o', 'w', 'e', 'r']
After merging 'l'+'o': ['lo', 'w', 'e', 'r']
After merging 'lo'+'w': ['low', 'e', 'r']
After merging 'e'+'r': ['low', 'er']
After merging 'low'+'er': ['lower']
```

In practice, these merges are learned from billions of tokens of text. Frequently occurring words (like "the") become single tokens early on, while rare words remain split into subwords. The result is an efficient encoding where common patterns use fewer tokens and rare patterns gracefully decompose into smaller pieces.

**Byte-level BPE** (used by GPT-2 and all subsequent OpenAI models) starts from the 256 possible byte values rather than Unicode characters. This guarantees that any input — including emojis, special characters, or text in any language — can always be tokenized. There are never "unknown" tokens.

### Tokenizers Used by Major LLMs

| Tokenizer | Used By | Vocabulary Size | Key Feature |
|-----------|---------|-----------------|-------------|
| **tiktoken** (BPE) | GPT-3.5/4, OpenAI models | ~100K–200K | Fast Rust implementation, byte-level BPE |
| **SentencePiece** (BPE/Unigram) | Llama, Mistral, T5 | 32K–128K | Language-agnostic, treats input as raw byte stream without pre-tokenization |
| **WordPiece** | BERT, DistilBERT | 30K | Uses likelihood-based merges rather than frequency-based |

### Why Tokenization Matters for Practitioners

- **Cost:** API-based LLMs charge per token. A more efficient tokenizer means the same text costs less to process. 
- **Context window:** A 128K token context window holds more actual text if the tokenizer is efficient. Poor tokenization of your data means you can fit less content in each request.
- **Multilingual performance:** Tokenizers trained primarily on English text may use 2–4× more tokens for the same content in other languages. This means non-English users get shorter effective context windows and pay more per word. Newer tokenizers include more multilingual tokens to reduce this gap.
- **Code and special content:** Code, mathematical notation, and structured data (like JSON) can tokenize very differently depending on the tokenizer. A single Python keyword might be one token or several.

### Tokenization in Practice

In [1]:
!pip install tiktoken transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.4 MB/s eta 0:00:0000:0100:01


In [2]:
import tiktoken

# Load the tokenizer used by GPT-4
enc = tiktoken.encoding_for_model("gpt-4")

text = "Large Language Models are transforming artificial intelligence."

# Encode text into tokens
tokens = enc.encode(text)
print(f"Original text: {text}")
print(f"Token IDs: {tokens}")
print(f"Number of tokens: {len(tokens)}")
print()

# Decode each token individually to see how the text was split
print("Individual tokens:")
for token_id in tokens:
    print(f"  {token_id} -> '{enc.decode([token_id])}'")

Original text: Large Language Models are transforming artificial intelligence.
Token IDs: [35353, 11688, 27972, 527, 46890, 21075, 11478, 13]
Number of tokens: 8

Individual tokens:
  35353 -> 'Large'
  11688 -> ' Language'
  27972 -> ' Models'
  527 -> ' are'
  46890 -> ' transforming'
  21075 -> ' artificial'
  11478 -> ' intelligence'
  13 -> '.'


In [3]:
# Compare tokenization across different text types
examples = [
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "supercalifragilisticexpialidocious",  # Rare word
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",  # Code
    "こんにちは世界",  # Japanese (non-English uses more tokens)
]

for text in examples:
    tokens = enc.encode(text)
    print(f"'{text}'")
    print(f"  -> {len(tokens)} tokens: {[enc.decode([t]) for t in tokens]}")
    print()

'Hello, world!'
  -> 4 tokens: ['Hello', ',', ' world', '!']

'The quick brown fox jumps over the lazy dog.'
  -> 10 tokens: ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog', '.']

'supercalifragilisticexpialidocious'
  -> 11 tokens: ['sup', 'erc', 'al', 'if', 'rag', 'il', 'istic', 'exp', 'ial', 'id', 'ocious']

'def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)'
  -> 23 tokens: ['def', ' fibonacci', '(n', '):', ' return', ' n', ' if', ' n', ' <=', ' ', '1', ' else', ' fibonacci', '(n', '-', '1', ')', ' +', ' fibonacci', '(n', '-', '2', ')']

'こんにちは世界'
  -> 4 tokens: ['こんにちは', '�', '�', '界']



---
# Modern Architectural Innovations

While all modern LLMs are built on the transformer architecture, they incorporate numerous improvements over the original design. These innovations address key challenges: making models faster, more memory-efficient, and capable of handling longer sequences. Understanding these building blocks is essential for understanding why modern LLMs perform so much better than their predecessors.

## Rotary Position Embeddings (RoPE)

In any transformer, the model needs some way to know the **order** of tokens in a sequence. Without positional information, the self-attention mechanism treats the input as a bag of tokens with no sense of which comes first, second, or last. The original transformer solved this with **absolute positional encodings** to signal "I am at position 5" or "I am at position 100."

### The Problem with Absolute Position Encodings
Absolute encodings have two significant limitations:
- They encode a fixed position ("I am token 5") rather than a **relative distance** ("I am 3 tokens away from that other token"). But in language, relative distance often matters more — the relationship between a pronoun and its antecedent depends on how far apart they are, not on their absolute positions in the document.
- They are fixed at training time. If a model is trained with positions 0–4095, it has never seen position 4096 and may behave unpredictably when asked to process longer sequences. This makes it hard to **extend** a model's context window after training.

### How RoPE Works

RoPE encodes position by **rotating** the query and key vectors in the attention mechanism. The idea comes from a mathematical insight where if you rotate vectors by angles proportional to their position, then when you compute the dot product between two rotated vectors, the result depends only on the **difference** in their positions.

Concretely, each pair of dimensions in the query and key vectors is treated as a 2D coordinate and rotated:

$$
f(x, m) = \begin{pmatrix} x_1 \cos(m\theta) - x_2 \sin(m\theta) \\ x_1 \sin(m\theta) + x_2 \cos(m\theta) \end{pmatrix}
$$

where $m$ is the position index and $\theta$ is a frequency that varies across dimension pairs (lower-numbered dimensions rotate slowly, higher-numbered dimensions rotate quickly).

**Why this is clever:** When the model computes the attention score (the dot product between a query at position $m$ and a key at position $n$), the rotation angles partially cancel out, and the result depends only on $m - n$ — the relative distance between the two tokens. This gives the model a natural, built-in sense of "how far apart are these two tokens?" without needing to learn it from scratch.

**Context extension:** Because RoPE uses continuous rotation angles, researchers found they can **interpolate** or **extrapolate** to positions far beyond what the model saw during training by adjusting the base frequency. Techniques like YaRN allow a model trained at 4K context to be extended to 128K or beyond with relatively little additional training.

**Used by:** Llama, Mistral, Qwen, PaLM, DeepSeek, and most modern open-weight LLMs.

## Grouped Query Attention (GQA)

Standard multi-head attention (as introduced in the original transformer) gives each attention head its own separate **Query**, **Key**, and **Value** projections. This means if you have 32 attention heads, you have 32 sets of Keys and 32 sets of Values. While this is great for model quality (each head can attend to different patterns), it creates a major **memory problem** during text generation.

### The KV Cache Problem

To understand why GQA matters, you need to understand the **KV cache** — one of the most important concepts in LLM inference.

When an LLM generates text, it produces one token at a time. For each new token, the model needs to compute attention over **all previous tokens** in the sequence. Without optimization, this would mean re-processing the entire sequence from scratch for every single new token.

The solution is the **KV cache**: after computing the Key and Value tensors for each token, we store them in memory so they can be reused. Each new token only needs to compute its own Query, then attend to the cached Keys and Values from all previous tokens.

The problem is that this cache grows with every token generated, and it must be stored for every layer and every attention head. For a model with:
- 32 layers, 32 attention heads, 128 dimensions per head, 8K context length

The KV cache requires: $32 \times 32 \times 128 \times 8192 \times 2 \times 2 = 4.3\text{ GB}$ (in FP16)

That is 4.3 GB of GPU memory **per sequence** just for the cache and this scales linearly with context length. At 128K context, it would be nearly 70 GB. This is often the primary bottleneck for serving LLMs.

### How GQA Solves This

**Grouped Query Attention** reduces the KV cache by sharing Key and Value heads across groups of Query heads. Instead of each of the 32 query heads having its own dedicated K/V head, groups of query heads share a single K/V head:

| Attention Type | Query Heads | Key/Value Heads | KV Cache Size |
|----------------|-------------|-----------------|---------------|
| **Multi-Head (MHA)** | 32 | 32 | 1× (baseline) |
| **Grouped Query (GQA)** | 32 | 8 | 0.25× |
| **Multi-Query (MQA)** | 32 | 1 | 0.03× |

With GQA at a 4:1 ratio, every 4 Query heads share the same Key and Value head. This reduces the KV cache by 4× with minimal impact on model quality. Research has shown that the Key and Value representations across heads are often quite similar, so sharing them loses surprisingly little information.

**Multi-Query Attention (MQA)** takes this to the extreme by having all query heads share a single K/V head. This maximizes memory savings but can hurt quality more noticeably. GQA is the sweet spot that most modern models have converged on.

**Used by:** Llama 2/3, Mistral, Gemma, CodeLlama

## Flash Attention

Standard self-attention computes a full $T \times T$ attention matrix (where $T$ is the sequence length), which is **quadratic** in both memory and compute. For a sequence of 8,192 tokens, that is a matrix with 67 million entries per head and per layer. For long sequences, this quickly becomes prohibitively expensive.

### The Bottleneck Is Memory, Not Math

Here is the key insight behind Flash Attention: the bottleneck in attention is **not** the number of arithmetic operations as modern GPUs can do enormous amounts of math very quickly. The real bottleneck is **memory bandwidth** or how fast data can be moved between different levels of GPU memory.

GPUs have two main types of memory:
- **HBM (High-Bandwidth Memory):** Large (40–80 GB on modern GPUs) but relatively slow to access. This is where model weights and the attention matrix normally live.
- **SRAM (on-chip cache):** Tiny (only a few MB) but extremely fast. This is where actual computations happen.

In standard attention, the full $T \times T$ attention matrix is computed in HBM, written out, then read back in for the softmax, then written out again, then read back in for the multiplication with the Value matrix. Each of these reads and writes is slow and the matrix is enormous.

### How Flash Attention Works

**Flash Attention** restructures the attention computation to avoid ever creating the full attention matrix:

1. **Tiling:** Instead of computing the entire $T \times T$ matrix at once, Flash Attention divides the Q, K, and V matrices into small blocks (tiles) that fit entirely in SRAM.
2. **Block-by-block computation:** For each tile, it computes the local attention scores, applies softmax incrementally (using a clever mathematical trick called the "online softmax"), and accumulates the result... all within fast SRAM.
3. **Fused operations:** The softmax normalization, masking, dropout, and matrix multiplication are all combined into a single GPU kernel, avoiding multiple slow round-trips to HBM.

The result is mathematically **identical** to standard attention, but because the computation stays in fast on-chip memory, it runs 2–4× faster and uses memory that scales **linearly** with sequence length instead of quadratically.

### Impact

Flash Attention has been so impactful that it is now the default attention implementation in virtually all LLM training and inference frameworks. It is integrated into PyTorch's `F.scaled_dot_product_attention`, Hugging Face Transformers, and all major serving frameworks. Flash Attention 2 and 3 further improved performance with better work partitioning across GPU threads and support for newer hardware features.

**Used by:** Nearly all modern LLM training and inference frameworks.

## Multi-Head Latent Attention (MLA)

While GQA reduces the KV cache by sharing Key/Value heads across groups, **DeepSeek-V3** introduced an even more aggressive approach: **Multi-Head Latent Attention (MLA)**.

### How MLA Works

Instead of caching full Key and Value vectors for each head, MLA compresses them into a small **latent vector** using a learned down-projection:

$$
c_t = W_{\text{down}} \cdot [K_t; V_t]
$$

At attention time, the latent vector is projected back up to reconstruct the Keys and Values:

$$
K_t = W_{\text{up}}^K \cdot c_t, \quad V_t = W_{\text{up}}^V \cdot c_t
$$

### Impact on KV Cache

| Method | KV Cache per Token | Reduction vs MHA |
|--------|-------------------|-----------------|
| **MHA** (standard) | $2 \times n_{\text{heads}} \times d_{\text{head}}$ | 1× (baseline) |
| **GQA** (4:1 ratio) | $2 \times \frac{n_{\text{heads}}}{4} \times d_{\text{head}}$ | 4× smaller |
| **MLA** (DeepSeek-V3) | $d_{\text{latent}}$ (much smaller) | ~**14×** smaller |

DeepSeek-V3 achieves approximately a **93% reduction** in KV cache compared to standard multi-head attention, enabling far more efficient long-context inference.

**Used by:** DeepSeek-V2, DeepSeek-V3

## Mixture of Experts (MoE)

One of the most impactful architectural innovations in recent LLMs is the **Mixture of Experts (MoE)** approach. It addresses a fundamental tension in model design: larger models perform better, but they are also slower and more expensive to run. MoE breaks this trade-off by allowing models to have a very large total parameter count while only **activating a fraction** of those parameters for any given input.

### The Core Idea

In a standard (dense) transformer, every token passes through **all** of the model's parameters at every layer. The feed-forward network (FFN), which makes up about two-thirds of the parameters in a typical transformer layer, processes every token identically.

In an MoE model, the single FFN is replaced with **multiple parallel FFNs** called "experts" (typically 8–256 per layer). A small **router network** (also called a gating network) examines each token and selects only the **top-K experts** (usually K=1 or K=2) to process that token. The other experts are completely skipped for that token.

### How It Works Step by Step

1. A token arrives at an MoE layer.
2. The router network (a small linear layer followed by softmax) looks at the token's hidden state and produces a probability distribution over all experts.
3. The top-K experts (typically K=2) with the highest router scores are selected.
4. Only those K experts process the token. Each expert is a standard FFN that produces its own output.
5. The outputs from the selected experts are combined using the router's probability scores as weights:

$$
y = \sum_{i \in \text{TopK}} g_i(x) \cdot E_i(x)
$$

where $g_i(x)$ is the router weight for expert $i$ and $E_i(x)$ is the output of expert $i$.

### Why This Is Powerful

The key insight is that **total parameters** and **active parameters** are decoupled:

| Aspect | Dense Model | MoE Model |
|--------|------------|----------|
| Total parameters | 70B | 671B |
| Active parameters per token | 70B | 37B |
| Quality | Baseline | Often better (more total knowledge stored) |
| Training compute per token | Proportional to total params | Proportional to active params |
| Inference speed | Slower (all 70B params used) | Faster (only 37B params used) |
| Memory required | 70B params | 671B params (all experts must be loaded) |

The model stores far more knowledge across all its experts (since it has more total parameters), but each individual token only uses a fraction of that knowledge (the most relevant experts). Think of it like a company with many specialized departments: for any given task, you only consult the relevant departments, but the company's total expertise is vast.

The **trade-off** is memory: all experts must be loaded into GPU memory even though only a few are active at any time. A 671B-parameter MoE model needs enough memory to hold all 671B parameters, even though it only computes with ~37B parameters per token.

### Notable MoE Models

- **Mixtral 8x7B** (Mistral AI, December 2023) — 8 experts per layer, 2 active per token. Total: ~47B params, active: ~13B. This model matched or beat Llama 2 70B (a dense model with 5× more active parameters), demonstrating the efficiency of MoE.
- **DeepSeek-V3** (December 2024) — 256 fine-grained experts per layer plus shared experts. Total: 671B params, active: 37B. Introduced auxiliary-loss-free load balancing and was trained remarkably cheaply (~$5.5M) using FP8 mixed-precision training.
- **GPT-4** is widely believed to use an MoE architecture, though this has not been officially confirmed by OpenAI.
- **Gemini 1.5** — Uses MoE to achieve efficient processing of its 1 million+ token context window.

### Load Balancing: The Key Challenge

A critical challenge in MoE is ensuring that tokens are distributed **roughly evenly** across experts. If the router learns to send most tokens to the same few "popular" experts, the other experts never receive enough training signal and effectively go to waste. This problem is called **expert collapse**.

Solutions include:
- **Auxiliary loss terms** — During training, an extra loss penalizes uneven expert usage, encouraging the router to spread tokens across all experts. The downside is that this extra loss can interfere with the main training objective.
- **Expert capacity limits** — A hard cap on how many tokens each expert can process per batch. Overflow tokens are dropped or sent to a default expert.
- **Auxiliary-loss-free balancing** (DeepSeek-V3) — Instead of an extra loss term, the router includes small learnable bias terms that are adjusted to keep expert utilization balanced without interfering with the main training signal. This is a cleaner approach that has shown strong results.

## Sliding Window Attention

Instead of allowing every token to attend to every other token (full attention), **sliding window attention** restricts each token to attend only to a fixed window of nearby tokens (e.g., the preceding 4,096 tokens).

### How It Works

- Each token at position $i$ can only attend to tokens in the range $[i - W, i]$, where $W$ is the window size.
- Information still propagates across the full sequence through **stacked layers**: after $L$ layers with window size $W$, a token can effectively attend to $L \times W$ tokens away.

### Benefits
- Reduces the computational cost of attention from $O(T^2)$ to $O(T \times W)$.
- Reduces KV cache memory since only $W$ past key-value pairs need to be stored.

**Used by:** Mistral 7B (window size 4,096), Mixtral, Gemma

## Context Window Evolution

The **context window** is the maximum number of tokens a model can process in a single forward pass — both the input you provide and the output it generates must fit within this window. Think of it as the model's "working memory": everything it can see and reason about at once.

Why does context length matter so much? Consider practical tasks:
- A single page of text is roughly **300-400 tokens**
- A 10-page research paper is around **3,000-4,000 tokens**
- An entire novel like *The Great Gatsby* is about **72,000 tokens**
- A large codebase with dozens of files can easily exceed **500,000 tokens**

With a 4K context window, you can barely fit a few pages of text plus your question. With a 1M+ window, you can load entire books, codebases, or hours of meeting transcripts and ask questions about any part of them.

The growth of context windows has been one of the most dramatic trends in LLM development:

| Model | Year | Context Window | Approx. Pages of Text |
|-------|------|---------------|----------------------|
| GPT-2 | 2019 | 1,024 tokens | ~3 pages |
| GPT-3 | 2020 | 2,048 tokens | ~5 pages |
| GPT-3.5 | 2022 | 4,096 tokens | ~10 pages |
| Claude 1 | 2023 | 9,000 tokens | ~23 pages |
| GPT-4 | 2023 | 8,192 / 32K tokens | ~20 / 80 pages |
| Claude 2 | 2023 | 100,000 tokens | ~250 pages |
| Llama 3.1 | 2024 | 128,000 tokens | ~320 pages |
| GPT-4o | 2024 | 128,000 tokens | ~320 pages |
| Claude 3.5 Sonnet | 2024 | 200,000 tokens | ~500 pages |
| Gemini 1.5 Pro | 2024 | 1,000,000 tokens | ~2,500 pages |
| Gemini 2.0 | 2025 | 1,000,000 tokens | ~2,500 pages |
| GPT-4.5 | 2025 | 128,000 tokens | ~320 pages |
| Claude 3.7 Sonnet | 2025 | 200,000 tokens | ~500 pages |
| Llama 4 Scout | 2025 | 10,000,000 tokens | ~25,000 pages |
| Llama 4 Maverick | 2025 | 1,000,000 tokens | ~2,500 pages |
| Claude Opus 4 | 2025 | 200,000 tokens | ~500 pages |
| Claude Sonnet 4 | 2025 | 200,000 tokens | ~500 pages |
| Gemini 2.5 Pro | 2025 | 1,000,000 tokens | ~2,500 pages |

A few notable observations from this trajectory:

1. **Exponential growth**: Context windows have grown roughly **10,000x** in just 6 years — from 1K tokens in GPT-2 to 10M tokens in Llama 4 Scout.

2. **The 128K–200K standard**: Most frontier models in 2024-2025 have settled on context windows in the 128K–200K range as the practical default for high-quality reasoning. This is enough to process entire books or large codebases.

3. **The million-token frontier**: Google's Gemini family and Meta's Llama 4 have pushed into the 1M–10M token range. Llama 4 Scout achieves its 10M-token context through an innovative **interleaved attention** pattern that alternates between local sliding window attention and full global attention across its 16 expert layers.

4. **Context length ≠ context quality**: A critical nuance — having a large context window doesn't mean the model uses all of it equally well. Research has shown a **"lost in the middle"** effect where models pay more attention to information at the beginning and end of their context, sometimes missing key details buried in the middle. The **Needle in a Haystack** test (placing a specific fact in a long document and asking the model to find it) has become a standard benchmark for measuring effective context utilization.

5. **Techniques enabling longer contexts**: Several innovations have made longer contexts practical:
   - **RoPE scaling** (position interpolation, NTK-aware scaling, YaRN) — extends position embeddings beyond training length
   - **Ring Attention** — distributes long sequences across multiple devices
   - **Landmark Attention** — compresses less important tokens to save memory
   - **Sliding window + global attention hybrids** — as used in Llama 4 and Mistral models

Even with these massive context windows, **Retrieval-Augmented Generation (RAG)** remains valuable — it's often more efficient to retrieve the 10 most relevant paragraphs from a million-document corpus than to try stuffing everything into the context window.

---
# The LLM Training Pipeline

Training a modern LLM is not a single step but rather it is a carefully orchestrated **multi-stage pipeline**. Each stage builds on the previous one, progressively transforming a randomly initialized neural network into a capable, helpful assistant. Understanding this pipeline is essential for understanding why LLMs behave the way they do, and why different models have different strengths and weaknesses.

## Stage 1: Pretraining

Pretraining is by far the most expensive and foundational stage. This is where the model learns the structure of language, acquires world knowledge, and develops basic reasoning abilities, all from the simple task of **predicting the next token**.

### The Objective: Next Token Prediction

The training objective is that when given a sequence of tokens, predict what comes next:

$$
L(\theta) = -\sum_{t=1}^{T} \log P(x_t \mid x_1, \ldots, x_{t-1}; \theta)
$$

In plain English: the model reads text left-to-right, and at each position, it tries to predict the next token. The loss function measures how surprised the model is by the actual next token; lower loss means better predictions.

Why does this simple objective produce such capable models? Because predicting the next token well requires understanding an enormous range of skills. To predict the next word in a sentence about physics, the model must understand physics. To predict the next line of code, it must understand programming. To predict the next move in a logical argument, it must understand logic. The breadth of the training data forces the model to become a generalist.

Over trillions of examples, the model learns:
- **Grammar and syntax** — How language is structured
- **World knowledge** — Facts about history, science, geography, current events
- **Reasoning patterns** — How to chain logical steps together
- **Code structure** — Programming languages, algorithms, debugging patterns
- **Mathematical relationships** — Arithmetic, algebra, symbolic manipulation
- **Social conventions** — How conversations work, tone, formality

### Training Data

The quality and composition of pretraining data has a dramatic impact on model capability. Modern pretraining datasets typically include:

| Source | Why It Is Included | Approximate Share |
|--------|-------------------|------------------|
| **Web crawls** (Common Crawl, filtered) | Broad world knowledge, diverse topics | 60–80% |
| **Books and academic papers** | Deep knowledge, formal reasoning | 5–10% |
| **Code** (GitHub, Stack Overflow) | Programming ability, structured thinking | 5–15% |
| **Wikipedia and reference material** | Factual accuracy, encyclopedic knowledge | 3–5% |
| **Curated high-quality sources** | Reasoning quality, writing quality | 5–10% |

Raw web data is extremely noisy being full of spam, duplicate content, low-quality writing, and toxic material. Before training, the data goes through an extensive cleaning pipeline: deduplication (removing near-identical pages), quality filtering (using classifiers to score text quality), PII removal, toxic content filtering, and domain-specific upsampling (training on more math and code to improve those capabilities).

### The Scale of Pretraining

Pretraining is extraordinarily expensive. To put it in perspective:
- **Llama 3 405B** used approximately 30.84 million GPU-hours on NVIDIA H100 GPUs... equivalent to running a single GPU continuously for over 3,500 years.
- **GPT-4** training cost is estimated at $50–100 million.
- Typical pretraining runs last **weeks to months** on clusters of thousands of GPUs working in parallel.

This massive compute requirement is why only a handful of organizations can train frontier LLMs from scratch.

## Stage 2: Supervised Fine-Tuning (SFT)

After pretraining, the model is an excellent **text completer** where if you give it the start of a sentence or paragraph, and it will continue in a plausible way. But it does not know how to be a helpful **assistant**. Ask it "What is the capital of France?" and instead of answering "Paris," it might generate five more geography questions (because in its training data, questions are often followed by more questions, as in quiz formats).

**Supervised Fine-Tuning** bridges this gap by training the model on curated examples of desired assistant behavior:

```
User: What is the capital of France?
Assistant: The capital of France is Paris. It is located in north-central France
along the Seine River and is the country's largest city with a population of
approximately 2.1 million in the city proper.
```

### How SFT Works

1. **Create a dataset** of high-quality prompt-response pairs (typically 10K–100K examples). These are often written by expert human annotators who craft ideal responses that are clear, accurate, well-structured, and appropriately detailed.

2. **Format examples** using a chat template that includes a system prompt (defining the assistant's behavior), user messages, and assistant responses. This teaches the model the conversational structure it should follow.

3. **Fine-tune** the pretrained model on this data using standard cross-entropy loss, but with a crucial detail: **the loss is only computed on the assistant's response tokens**, not on the prompt tokens. The model already knows how to read and understand prompts but what it needs to learn is how to *respond* to them appropriately.

### What SFT Teaches the Model

The transformation from base model to SFT model is dramatic:
- **Instruction following** — Understanding and executing diverse requests ("Write a poem," "Summarize this text," "Debug this code")
- **Conversational structure** — Responding in a helpful, well-organized format rather than just continuing text
- **Refusal behavior** — Declining inappropriate or harmful requests politely
- **Tone and style** — Adopting the helpful, knowledgeable tone expected of an assistant

### Quality Over Quantity

A key finding from research (notably the LIMA paper: "Less Is More for Alignment") is that **data quality matters far more than quantity** for SFT. A small dataset of 1,000 expert-written, carefully curated responses can produce a better assistant than 100,000 mediocre ones. The model already has vast knowledge from pretraining; SFT just needs to teach it the *format* of being helpful, not new knowledge.

## Stage 3: Preference Optimization

After SFT, the model can follow instructions and produce well-structured responses. But it may not produce responses that humans actually **prefer**. It might be overly verbose when a short answer would be better, miss the most important point of a question, hedge excessively, or produce subtly unhelpful answers that are technically correct but practically useless.

The challenge is that "what makes a good response" is extremely difficult to specify in a loss function. It involves subjective judgments about helpfulness, clarity, tone, completeness, and safety that vary by context. **Preference optimization** addresses this by learning directly from human judgments about which responses are better.

### RLHF (Reinforcement Learning from Human Feedback)

RLHF was the technique that powered ChatGPT's breakthrough moment in November 2022. It is a three-step process:

**Step 1: Collect comparison data.**  
Human raters are shown a prompt and two or more model responses. They rank the responses from best to worst. For example:

```
Prompt: "Explain quantum entanglement simply."

Response A: "Quantum entanglement is a phenomenon where two particles 
become correlated such that the quantum state of one instantly 
influences the other, regardless of distance."  [Ranked #2]

Response B: "Imagine you have two magic coins. When you flip one and 
get heads, the other always lands on tails — no matter how far apart 
they are. That's essentially what quantum entanglement is..."  [Ranked #1]
```

**Step 2: Train a reward model.**  
A separate neural network (often initialized from the SFT model) learns to predict which responses humans prefer. Given a (prompt, response) pair, it outputs a single scalar reward score. Higher scores mean the response is more likely to be preferred. This reward model essentially captures "what humans consider a good response" in a learnable function.

**Step 3: Optimize the LLM with reinforcement learning.**  
Using Proximal Policy Optimization (PPO), the LLM is trained to generate responses that maximize the reward model's score. A critical constraint is added — a **KL divergence penalty** that prevents the model from straying too far from the SFT model:

$$
\max_{\pi_\theta} \; \mathbb{E}_{x \sim D, \, y \sim \pi_\theta(y|x)} \left[ R(x, y) - \beta \, \text{KL}\left(\pi_\theta(y|x) \| \pi_{\text{ref}}(y|x)\right) \right]
$$

Without this KL penalty, the model would quickly learn to "hack" the reward model and produce outputs that score highly on the reward metric but are actually degenerate (e.g., repetitive phrases that the reward model happens to rate well). The KL penalty keeps the model grounded in its SFT training.

### DPO (Direct Preference Optimization)

**DPO** is an elegant simplification of RLHF that eliminates the need for both the separate reward model and the unstable RL training loop. The key mathematical insight is that the optimal policy in the RLHF framework can be expressed as a closed-form function of the preference data itself.

Instead of the three-step RLHF process, DPO directly optimizes the LLM using pairs of preferred and rejected responses:

$$
L_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma\left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \right]
$$

where $y_w$ is the preferred ("winning") response and $y_l$ is the less preferred ("losing") response. In plain English, this loss function says: "Increase the probability of the preferred response and decrease the probability of the rejected response, relative to the reference model."

**Why DPO became so popular:**
- No reward model to train and maintain
- No RL training loop (which is notoriously unstable and hard to tune)
- Just a standard supervised loss on preference pairs which can run in any training framework
- Achieves comparable results to RLHF in most settings

### Constitutional AI (Anthropic)

Anthropic's approach to alignment, used to train Claude, addresses a practical problem with RLHF: it requires **humans to read and evaluate potentially harmful content** to produce comparison data for safety training. This is both expensive and psychologically taxing.

Constitutional AI replaces much of this human feedback with **AI feedback**:

1. The model generates responses to potentially harmful prompts.
2. The model is asked to **critique** its own response against a set of written principles (the "constitution") — for example, "Is this response helpful? Could it cause harm? Does it respect privacy?"
3. The model **revises** its response based on its own critique.
4. The (original, revised) pairs are used as preference data for training.

This approach scales the alignment process (AI can evaluate millions of responses), makes it more consistent (the same principles are applied every time), and makes the alignment goals **transparent and auditable** (the constitution is a readable document).

### Comparison of Approaches

| Method | Reward Model? | RL Training? | Stability | Simplicity |
|--------|-------------|-------------|-----------|------------|
| **RLHF** | Yes (separate model) | Yes (PPO) | Can be unstable | Complex |
| **DPO** | No | No | More stable | Simple |
| **Constitutional AI** | AI-generated | Yes | Scalable | Moderate |

### The Full Training Pipeline Visualized

```
Random Weights
     │
     ▼
┌─────────────────────┐
│   PRETRAINING        │  Trillions of tokens from the internet
│   (Next Token Pred.) │  Cost: $10M–$100M+
└─────────┬───────────┘
          │
          ▼
    Base Model (can complete text, but not chat)
          │
          ▼
┌─────────────────────┐
│   SFT                │  10K–100K expert-written examples
│   (Instruction       │  Cost: $10K–$1M
│    Following)        │
└─────────┬───────────┘
          │
          ▼
    SFT Model (follows instructions, but may not match preferences)
          │
          ▼
┌─────────────────────┐
│   RLHF / DPO         │  Human preference comparisons
│   (Preference         │  Cost: $100K–$10M
│    Alignment)         │
└─────────┬───────────┘
          │
          ▼
    Aligned Model (helpful, harmless, honest)
```

---
# Scaling Laws

One of the most important discoveries in LLM research is that model performance follows **predictable scaling laws**. These laws relate performance (measured by loss on a test set) to three factors:

1. **Model size** (number of parameters, $N$)
2. **Dataset size** (number of training tokens, $D$)
3. **Compute budget** (total FLOPs, $C$)

### The Chinchilla Scaling Laws

The Chinchilla paper (Hoffmann et al., 2022) from DeepMind showed that many LLMs were **undertrained** such that they used too many parameters for the amount of data they were trained on.

**Chinchilla's key finding:** For a given compute budget, the optimal model has roughly **20 tokens of training data per parameter**.

| Model | Parameters | Training Tokens | Tokens/Parameter | Chinchilla-Optimal? |
|-------|-----------|----------------|-------------------|-------------------|
| GPT-3 | 175B | 300B | 1.7 | Undertrained |
| Chinchilla | 70B | 1.4T | 20 | Yes |
| Llama 2 70B | 70B | 2T | 29 | Overtrained (intentionally) |
| Llama 3 8B | 8B | 15T | 1,875 | Heavily overtrained |

**Why overtrain?** The Chinchilla scaling law optimizes for training compute. But in practice, a smaller model that is trained on more data is **cheaper to serve** at inference time. Since inference costs dominate in production, companies now deliberately overtrain smaller models.

### Emergent Abilities and the Debate Around Them

Some capabilities appear to emerge suddenly at certain model scales rather than improving gradually:
- **Few-shot learning** — The ability to learn from examples in the prompt
- **Chain-of-thought reasoning** — Working through problems step by step
- **Code generation** — Writing functional code from descriptions

However, this picture has been challenged. **Schaeffer et al. (2023)** argued that "emergence" may largely be a **measurement artifact**:
- When using **discontinuous metrics** (like exact-match accuracy: either 100% right or 0%), performance appears to jump suddenly at a certain scale.
- When using **continuous metrics** (like per-token log-likelihood or edit distance), performance actually improves **smoothly and predictably** across all scales.

**The current understanding:** The underlying capabilities likely improve smoothly with scale, but their **practical utility** can still be threshold-like. A model that gets 30% of a math problem's steps right is useless; one that gets 95% right is useful. The "emergence" is in crossing the threshold of practical usefulness, not in a sudden internal change.

## Inference-Time Scaling (Reasoning Models)

One of the most significant recent developments in LLMs is the idea of **scaling compute at inference time** — spending more time "thinking" about harder problems rather than always generating answers at the same speed.

### The Traditional Approach

Standard LLMs generate every response with roughly the same amount of compute, regardless of difficulty. Asking "What is 2+2?" and "Prove the Riemann hypothesis" both take one forward pass per token.

### The Reasoning Model Paradigm

**Reasoning models** (OpenAI's o1/o3, DeepSeek-R1) change this by generating a long internal **chain of thought** before producing a final answer. The model:

1. Breaks the problem into steps.
2. Explores multiple reasoning paths.
3. Verifies and self-corrects intermediate results.
4. Only then produces a final answer.

This means harder problems get more compute (longer chains of thought), while easy problems can be answered quickly.

### How Reasoning Models Are Trained

**DeepSeek-R1** revealed a particularly interesting training approach called **GRPO (Group Relative Policy Optimization)**:

1. For each prompt, generate a **group of candidate responses**.
2. Score them with an automated verifier (e.g., checking if a math answer is correct).
3. Use the **relative ranking within the group** as the reward signal — no separate reward model needed.

$$
\text{reward}(y_i) = \frac{r(y_i) - \text{mean}(r(y_1, \dots, y_G))}{\text{std}(r(y_1, \dots, y_G))}
$$

This is simpler than RLHF (no reward model) and simpler than DPO (no paired preferences needed). DeepSeek-R1-Zero showed that pure RL training, without any SFT, can produce emergent reasoning behaviors like self-verification and reflection.

### Why This Is a New Scaling Dimension

| Scaling Dimension | What Scales | When It Helps |
|-------------------|------------|---------------|
| **Pretraining compute** | Model size + training data | General capability |
| **Inference-time compute** | Chain-of-thought length | Hard reasoning, math, code |

This means that even without making models bigger, we can improve performance on hard tasks by letting models "think longer." This is a fundamental shift in how we think about LLM capability.

---
# Fine-Tuning LLMs

While pretraining creates a general-purpose model, **fine-tuning** adapts it to specific tasks, domains, or behaviors. A general-purpose LLM might be decent at everything but excellent at nothing in particular. Fine-tuning on medical literature can create a medical specialist; fine-tuning on legal documents can create a legal assistant; fine-tuning on a company's internal data can create a custom support agent.

Modern fine-tuning techniques have made this process accessible even with limited hardware, which is why fine-tuning has become one of the most important practical skills in working with LLMs.

## Full Fine-Tuning

In full fine-tuning, **all parameters** of the model are updated during training. This gives maximum flexibility — every weight in the network can be adjusted — but it is extremely resource-intensive:

- The model weights must be stored in memory (e.g., ~16 GB for an 8B model in FP16).
- The **optimizer states** (Adam stores two additional values per parameter: momentum and variance) require roughly 2× the model size.
- **Gradients** require another 1× the model size.
- **Activations** stored for backpropagation add more on top.

In total, full fine-tuning requires approximately **4–6× the model size** in GPU memory. A Llama 3 8B model requires ~64 GB just for fine-tuning — more than any single consumer GPU. For 70B models, you would need a cluster of high-end GPUs.

For most practitioners, full fine-tuning of large models is impractical. This is where parameter-efficient methods come in.

## LoRA (Low-Rank Adaptation)

**LoRA** (Hu et al., 2021) is the most popular parameter-efficient fine-tuning method, and understanding how it works requires one key insight: **the weight changes during fine-tuning are surprisingly low-dimensional**.

When you fine-tune a model, you are computing a weight update $\Delta W$ — the difference between the original weights and the fine-tuned weights. Research has shown that this update matrix, despite being very large (e.g., 4096 × 4096 = 16.8 million entries), can be well-approximated by a **low-rank matrix**. In other words, the "useful information" in the fine-tuning update lives in a much smaller space than the full weight matrix.

### How LoRA Works

For a pretrained weight matrix $W_0 \in \mathbb{R}^{d \times k}$, instead of learning a full update $\Delta W$, LoRA decomposes it into two small matrices:

$$
W = W_0 + \Delta W = W_0 + BA
$$

where:
- $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$ are the trainable LoRA matrices
- $r \ll \min(d, k)$ is the **rank** (typically 8–64)
- $W_0$ is **completely frozen** — its weights never change
- Only $A$ and $B$ are trained

**Why does this work?** The matrix $BA$ has rank at most $r$. If $r = 16$ and $d = k = 4096$, then instead of learning 16.8 million parameters, you only learn $4096 \times 16 + 16 \times 4096 = 131K$ parameters — a **128× reduction**. Yet experiments consistently show that this small adapter captures the essential information needed for fine-tuning, achieving results comparable to full fine-tuning.


### Practical Advantages of LoRA

- **Memory efficient** — Only the small adapter matrices need gradients and optimizer states. The frozen base model just sits in memory (and can even be quantized — see QLoRA below).
- **Composable** — You can train multiple LoRA adapters for different tasks (one for medical, one for legal, one for code) and swap them in and out at inference time without reloading the base model. This is like having different "skill modules" you can plug into the same base model.
- **No inference overhead** — After training, the adapter weights can be mathematically merged back into the base model: $W = W_0 + BA$. The merged model runs at exactly the same speed as the original — no additional computation.
- **Small file sizes** — A LoRA adapter for a 70B model might be only 50–200 MB, compared to 140 GB for the full model. This makes sharing and distributing fine-tuned models practical.

### Where LoRA Is Applied

LoRA adapters are typically added to the **attention projection matrices** (Q, K, V, and output projections) in each transformer layer. Some practitioners also apply LoRA to the feed-forward network layers. The choice of which layers to adapt and what rank to use are the main hyperparameters to tune.

## QLoRA (Quantized LoRA)

**QLoRA** (Dettmers et al., 2023) combines LoRA with aggressive model quantization, making fine-tuning accessible on consumer hardware:

1. **Quantize** the base model to 4-bit precision using NF4 (Normal Float 4) — a data type specifically designed for the weight distributions found in neural networks. This reduces the base model's memory footprint by ~4×.
2. **Add LoRA adapters** in full precision (BF16) on top of the quantized model. The adapters need full precision for stable training, but they are tiny.
3. **Train only the LoRA adapters** while the 4-bit base model stays completely frozen.

The key innovation is that during the backward pass, gradients flow through the quantized weights (which are dequantized on-the-fly for computation), but only the LoRA parameters are actually updated. This makes it possible to fine-tune a **70B parameter model on a single 48GB GPU** — something that would otherwise require a cluster of 8+ GPUs with full fine-tuning.

### Fine-Tuning with LoRA in Practice

Below is an example of fine-tuning a small language model using LoRA with Hugging Face's PEFT library.

In [4]:
!pip install peft transformers datasets accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 4.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 13.0 MB/s eta 0:00:0000:0100:01
  Using cached torch-2.2.2-cp310-none-macosx_10_9_x86_64.whl (150.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.8/467.8 kB 13.3 MB/s eta 0:00:00
  Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 19.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 39.1 MB/s eta 0:00:0000:0100:01
  Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.4.0
    Uninstalling typing_extensions-4.4.0:
      Successfully uninstalled typing_extensions-4.4.0
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.3.2


    Uninstalling safetensors-0.3.2:
      Successfully uninstalled safetensors-0.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.10.0 requires libclang>=13.0.0, which is not installed.
tensorflow 2.10.0 requires tensorflow-io-gcs-filesystem>=0.23.1, which is not installed.
tensorflow 2.10.0 requires protobuf<3.20,>=3.9.2, but you have protobuf 3.20.3 which is incompatible.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

# Load a small model for demonstration
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Base model parameters: {sum(p.numel() for p in model.parameters()):,}")

/Users/jonathanschlosser/anaconda3/envs/DeepLearningEnv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,      # Task type: causal language modeling
    r=16,                               # Rank of the low-rank matrices
    lora_alpha=32,                      # Scaling factor (higher = larger updates)
    lora_dropout=0.1,                   # Dropout for regularization
    target_modules=["c_attn", "c_proj"] # Which layers to apply LoRA to
)

# Wrap the model with LoRA adapters
model = get_peft_model(model, lora_config)

# Print trainable vs total parameters
model.print_trainable_parameters()

In [ ]:
# Load a small dataset for demonstration
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1000]")

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"Training examples: {len(tokenized_dataset)}")

In [ ]:
# Set up training
training_args = TrainingArguments(
    output_dir="./lora-gpt2",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Train
trainer.train()

In [ ]:
# Generate text with the fine-tuned model
model.eval()
prompt = "The history of artificial intelligence"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

---
# Inference Optimizations

Training an LLM is expensive, but the **ongoing cost of running it** (inference) is what dominates in production. A company serving millions of users makes far more inference calls than training runs. A 70B parameter model in FP16 requires ~140 GB of GPU memory just to load — more than any single consumer GPU can hold. Several techniques make LLM inference practical and affordable.

## Quantization

**Quantization** reduces the numerical precision of model weights from 16-bit (or 32-bit) floating point to lower bit widths. The core idea is straightforward: instead of storing each weight as a 16-bit number (which can represent ~65,000 distinct values), store it as a 4-bit number (which can represent only 16 distinct values). You lose some precision, but the model still works surprisingly well because neural networks are remarkably robust to small perturbations in their weights.

| Precision | Bits per Parameter | Memory for 70B Model | Quality Impact |
|-----------|-------------------|---------------------|----------------|
| FP32 | 32 | 280 GB | Baseline (training precision) |
| FP16/BF16 | 16 | 140 GB | Negligible — standard inference |
| INT8 | 8 | 70 GB | Very small |
| INT4 (GPTQ/AWQ) | 4 | 35 GB | Small but measurable |
| GGUF Q4_K_M | ~4.8 | ~42 GB | Small — good quality/size trade-off |

### Popular Quantization Formats

- **GPTQ** (GPT Quantization) — A post-training method that quantizes weights to 4-bit by running a small calibration dataset through the model and adjusting the quantized weights layer-by-layer to minimize the output error. It uses second-order (Hessian) information to decide which weights can tolerate more quantization and which need more precision. Runs on GPU with custom CUDA kernels for fast inference.

- **AWQ** (Activation-Aware Weight Quantization) — A more recent approach that observes which weight channels correspond to large activation values and protects those "salient" channels by scaling them up before quantization. The insight is that a small fraction (~1%) of weights disproportionately affect the output, so preserving those precisely matters more than treating all weights equally. Generally produces better quality than GPTQ at the same bit width.

- **GGUF** (used by llama.cpp) — A flexible quantization format designed for **CPU inference**. It supports mixed precision (different layers can use different bit widths based on their sensitivity) and runs efficiently without a GPU. This is what makes it possible to run a 7B model on a MacBook or a 70B model on a high-end gaming PC. The various GGUF quantization levels (Q2_K through Q8_0) represent different quality-vs-size trade-offs, with "K" variants using importance-aware quantization that allocates more bits to sensitive layers.

## Speculative Decoding

LLMs generate tokens **one at a time**, and each token requires a full forward pass through the model. This is inherently sequential and slow. **Speculative decoding** is a clever technique that speeds this up without changing the output quality at all.

### How It Works

1. A small, fast **draft model** (e.g., a 1B parameter model) generates several candidate tokens quickly. Because it is small, each forward pass is very fast.
2. The large **target model** (the model you actually want to use) takes all of the draft model's candidate tokens and **verifies them in a single forward pass**. This works because attention can process multiple tokens in parallel — the bottleneck is only in sequential generation, not parallel verification.
3. Tokens are accepted from left to right as long as the target model "agrees" (would have assigned sufficient probability to the same token). On the first rejection, the target model's own distribution is used to sample a replacement.

The critical insight: the target model's forward pass takes roughly the same time whether it is verifying 1 token or 8 tokens (since GPU compute is parallelized). So if the draft model guesses 6 tokens correctly and 2 are rejected, you effectively got 6 tokens for the cost of ~1 target model forward pass, achieving a **2–3× speedup** with mathematically identical output distribution.

## Serving Frameworks

Efficiently serving LLMs to many users simultaneously requires specialized software:

| Framework | Key Innovation | Best For |
|-----------|---------------|---------|
| **vLLM** | PagedAttention — manages KV cache like virtual memory pages, eliminating memory fragmentation and waste | High-throughput GPU serving |
| **TGI** (Hugging Face) | Continuous batching + tensor parallelism, tight Hugging Face integration | Production serving with HF models |
| **llama.cpp** | CPU inference with GGUF quantization, minimal dependencies | Running models on laptops and consumer hardware |
| **Ollama** | User-friendly wrapper around llama.cpp with model management | Easy local model deployment for individuals |

**PagedAttention** (the key innovation in vLLM) deserves special mention: in naive KV cache management, memory is allocated in contiguous blocks for each sequence, leading to significant fragmentation and waste (typically 60–80% of allocated KV cache memory is wasted). PagedAttention manages KV cache in small, non-contiguous pages — like how an operating system manages RAM with virtual memory. This nearly eliminates waste and enables 2–4× higher throughput than naive serving.

### Running a Quantized Model Locally

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Use 4-bit quantization
    bnb_4bit_quant_type="nf4",            # NormalFloat4 quantization type
    bnb_4bit_compute_dtype=torch.bfloat16 # Compute in bfloat16 for speed
)

# Load a model in 4-bit precision
# Note: This requires a GPU with bitsandbytes installed
model_name = "gpt2"  # Using GPT-2 for demonstration; replace with larger models as needed
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

# Check the model's memory footprint
print(f"Model memory footprint: {model.get_memory_footprint() / 1e6:.1f} MB")

# Generate text
inputs = tokenizer("Artificial intelligence will", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

---
# Retrieval-Augmented Generation (RAG)

LLMs have a fundamental limitation: their knowledge is **frozen at training time**. Once training is complete, the model cannot learn new information, access private documents, or check real-time data. If you ask about something that happened after the training cutoff, the model either does not know or — worse — confidently generates a plausible-sounding but incorrect answer (a **hallucination**).

**Retrieval-Augmented Generation (RAG)** solves this by combining an LLM with an external knowledge retrieval system. Instead of relying solely on what the model memorized during training, RAG retrieves relevant information from an external knowledge base and provides it as context for the model to use when generating its response.

### The Three Problems RAG Solves

- **Knowledge cutoff:** LLMs only know what was in their training data. A model trained in early 2024 cannot answer questions about events in late 2024. RAG gives the model access to up-to-date information.
- **Hallucination:** When asked about topics not well-covered in training data, LLMs may generate plausible-sounding but incorrect information. RAG grounds the model's responses in actual documents, dramatically reducing hallucination.
- **Private data:** LLMs have no knowledge of an organization's internal documents, databases, customer records, or proprietary information. RAG provides this context at query time without needing to fine-tune the model.

### How RAG Works

The RAG pipeline has four main steps:

```
User Query: "What was our Q4 revenue?"
        │
        ▼
┌──────────────────┐
│  1. EMBED QUERY   │  Convert the query into a dense vector (a list of numbers
│                    │  that captures its meaning) using an embedding model
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│  2. RETRIEVE      │  Search a vector database for document chunks whose
│     DOCUMENTS     │  embeddings are closest to the query embedding.
│                    │  Returns the top-K most semantically similar passages.
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│  3. AUGMENT       │  Insert the retrieved passages into the LLM's prompt
│     PROMPT        │  as additional context: "Based on the following documents,
│                    │  answer the question: [docs] [question]"
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│  4. GENERATE      │  The LLM generates an answer grounded in the retrieved
│     RESPONSE      │  documents, citing specific information from them
└──────────────────┘
```

### Key Components Explained

**1. Embedding Model**

An embedding model converts text into **dense vectors** (lists of floating-point numbers, typically 384–3072 dimensions) where **semantically similar texts are close together** in the vector space. For example, "What is the capital of France?" and "Paris is the capital city of France" would have very similar embedding vectors, even though they share few exact words.

This is different from keyword search: embedding-based search understands meaning, not just word overlap. A query about "machine learning optimization" would match documents about "training neural networks" even if those exact words never appear.

Popular embedding models: OpenAI `text-embedding-3-small`, `sentence-transformers/all-MiniLM-L6-v2`, Cohere Embed

**2. Vector Database**

A vector database is a specialized database optimized for storing millions of embedding vectors and finding the most similar ones quickly. Standard databases use exact matching (find rows where `city = 'Paris'`); vector databases use **approximate nearest neighbor (ANN)** search to find the vectors closest to a query vector in high-dimensional space.

Popular choices: Pinecone (managed cloud), Weaviate (open-source), Chroma (lightweight/local, great for prototyping), FAISS (Meta's library), Qdrant (Rust-based, performant), pgvector (PostgreSQL extension — adds vector search to your existing database)

**3. Chunking Strategy**

Before embedding, documents must be split into smaller pieces (chunks) because:
- Embedding models have input length limits (typically 512–8,192 tokens)
- **Smaller chunks enable more precise retrieval.** If a 100-page document is stored as one chunk, the entire document gets retrieved for any related query. If it is split into paragraph-sized chunks, only the most relevant paragraphs are retrieved.

Common approaches:
- **Fixed-size chunks with overlap:** Split every 512 tokens, with 50 tokens of overlap between adjacent chunks to avoid cutting sentences mid-thought.
- **Sentence-based splitting:** Split at natural sentence boundaries, grouping sentences into chunks of a target size.
- **Semantic chunking:** Use an embedding model to detect topic shifts within a document and chunk at those boundaries, keeping topically coherent sections together.

**4. The LLM**

The LLM receives the original query plus the retrieved context and generates a response grounded in the provided documents. The prompt typically instructs the model to only use information from the provided context and to say "I don't know" if the context does not contain the answer — this reduces hallucination.

### Simple RAG Example

In [ ]:
!pip install chromadb sentence-transformers

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create a simple document collection
documents = [
    "The transformer architecture was introduced in the 2017 paper 'Attention Is All You Need' by Vaswani et al.",
    "GPT-4 was released by OpenAI in March 2023 and is believed to use a Mixture of Experts architecture.",
    "LoRA is a parameter-efficient fine-tuning method that adds low-rank matrices to frozen pretrained weights.",
    "Flash Attention reduces the memory complexity of attention from quadratic to linear by using tiling.",
    "RLHF uses human preference data to train a reward model, which then guides LLM optimization via PPO.",
    "Retrieval-Augmented Generation combines LLMs with external knowledge retrieval to reduce hallucinations.",
    "DeepSeek-V3 uses 671 billion total parameters with only 37 billion active per token via MoE.",
    "Llama 3 was trained on 15 trillion tokens and released in sizes from 8B to 405B parameters."
]

# Create vector store
client = chromadb.Client()
collection = client.create_collection("llm_knowledge")

# Add documents to the collection
collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))]
)

print(f"Added {len(documents)} documents to the vector store.")

In [ ]:
# Query the vector store
query = "How does LoRA work for fine-tuning?"

results = collection.query(
    query_texts=[query],
    n_results=3  # Retrieve top 3 most relevant documents
)

print(f"Query: {query}\n")
print("Retrieved documents (ranked by relevance):")
for i, doc in enumerate(results["documents"][0]):
    print(f"  {i+1}. {doc}")

# In a full RAG pipeline, these documents would be inserted into an LLM prompt:
context = "\n".join(results["documents"][0])
rag_prompt = f"""Based on the following context, answer the question.

Context:
{context}

Question: {query}

Answer:"""

print(f"\n--- RAG Prompt that would be sent to the LLM ---\n{rag_prompt}")

### Advanced RAG Patterns

As RAG has matured, several advanced patterns have emerged for production systems:

**Reranking**
After initial retrieval returns the top-K candidates, a **cross-encoder reranker** (e.g., Cohere Rerank, BGE-Reranker) scores each query-document pair more carefully. This is slower but significantly more accurate than the initial bi-encoder similarity search.

```
Query → Bi-encoder retrieval (fast, top-50) → Cross-encoder reranking (accurate, top-5) → LLM
```

**HyDE (Hypothetical Document Embeddings)**
Instead of embedding the raw query, first ask the LLM to generate a **hypothetical answer**, then embed *that* for retrieval. The intuition: a hypothetical answer is more semantically similar to actual relevant documents than a short question.

```
Query: "How does LoRA work?"
     ↓ LLM generates hypothetical answer
"LoRA adds low-rank matrices BA alongside frozen weights W, where rank r << d..."
     ↓ Embed this instead of the original query
Retrieve documents similar to the hypothetical answer
```

**Agentic RAG**
Instead of a fixed retrieve-then-generate pipeline, let the **agent decide** when and what to retrieve. The agent might:
- Break a complex query into sub-queries and retrieve for each
- Look at initial results and decide more information is needed
- Route different parts of a question to different knowledge bases

**GraphRAG**
Build a **knowledge graph** from documents (entities as nodes, relationships as edges), then combine graph traversal with vector search. This helps with questions that require understanding relationships across multiple documents — something flat vector search struggles with.

---
# Multimodal LLMs

Early LLMs were **text-only** — they could read and write text, but had no ability to see images, hear audio, or watch video. Modern LLMs are increasingly **multimodal**, meaning they can process and reason about multiple types of input. This represents one of the most important recent advances in AI, because the real world is inherently multimodal — we communicate through text, images, diagrams, speech, and video simultaneously.

## How Vision-Language Models Work

Adding vision to an LLM is not as simple as "showing" it an image. The model's transformer architecture processes sequences of tokens — discrete numerical vectors. Images need to be converted into a similar format. The most common approach involves three components working together:

**1. Vision Encoder**

A pretrained image model (typically a **Vision Transformer**, or ViT, often from OpenAI's CLIP or Google's SigLIP) processes the image. It divides the image into a grid of patches (e.g., 14×14 pixel patches), treats each patch as a "visual token," and processes them through transformer layers to produce a sequence of visual feature vectors — rich representations that capture what is in different parts of the image.

**2. Projection Layer**

The visual features from the vision encoder live in a different representation space than the LLM's text token embeddings. A **projection layer** (which can be as simple as a two-layer MLP or as complex as a cross-attention module) maps the visual features into the LLM's embedding space. After this projection, the visual tokens "look like" text tokens to the LLM — they are vectors of the same dimensionality and in the same learned space.

**3. LLM Backbone**

The language model processes the projected visual tokens alongside text tokens as if they were all part of the same sequence. From the LLM's perspective, it is just processing a sequence of tokens — some happen to represent text and some happen to represent image patches. The attention mechanism naturally learns to cross-reference between visual and text tokens.

```
Image  ──→  [Vision Encoder]  ──→  [Projection]  ──→  Visual Tokens
                                                          │
Text   ──→  [Tokenizer]      ──→  [Embedding]    ──→  Text Tokens
                                                          │
                                                     ┌────┴────┐
                                                     │   LLM   │
                                                     │Backbone │
                                                     └────┬────┘
                                                          │
                                                     Text Output
```

### Training a Vision-Language Model

Training typically happens in two stages:

1. **Vision-language pretraining:** Train the projection layer on large datasets of image-caption pairs (millions of images with text descriptions). The vision encoder and LLM are usually frozen at this stage. The projection layer learns to map visual features into the LLM's space so that the LLM can "understand" what it sees.

2. **Visual instruction tuning:** Fine-tune the full model (or at least the LLM and projection layer) on visual question-answering, chart understanding, document analysis, and other visual reasoning tasks. This teaches the model not just to describe what it sees, but to reason about it and follow complex instructions involving images.

### Notable Multimodal Models

| Model | Developer | Modalities | Key Innovation |
|-------|-----------|-----------|----------------|
| **GPT-4V/4o** | OpenAI | Text, images, audio | GPT-4o is natively multimodal — a single model processes all modalities, not separate models piped together. This enables faster, more natural interactions. |
| **Claude 3/3.5/4** | Anthropic | Text, images, PDFs | Particularly strong at understanding documents, charts, diagrams, and screenshots — practical for business use cases. |
| **Gemini 1.5/2** | Google | Text, images, video, audio | Can process up to 1M tokens of context including hours of video. Native video understanding without frame extraction. |
| **LLaVA** | Open-source | Text, images | Demonstrated that simple visual instruction tuning on a modest dataset can produce surprisingly capable vision-language models. Influential for its simplicity. |
| **Pixtral** | Mistral AI | Text, images | Open-weight multimodal model from Mistral, making vision-language capabilities accessible to the open-source community. |

### Why Multimodality Matters

The practical applications are vast and growing:
- **Document understanding** — Analyzing PDFs, invoices, charts, diagrams, and screenshots without needing OCR or manual data extraction
- **Visual reasoning** — Answering questions about images, comparing visual elements, identifying anomalies
- **Code from mockups** — Converting UI design mockups and wireframes into functional HTML/CSS/code
- **Accessibility** — Generating detailed image descriptions for visually impaired users
- **Scientific analysis** — Interpreting experimental plots, microscopy images, medical scans, satellite imagery

### Using a Multimodal Model (GPT-4 Vision)

In [ ]:
from openai import OpenAI
import base64

api_key = ''  # Add your OpenAI API key here
client = OpenAI(api_key=api_key)

# Example: Analyze an image using GPT-4 Vision
# You can pass an image URL or a base64-encoded image
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What is shown in this image? Describe it in detail."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
                    }
                }
            ]
        }
    ],
    max_tokens=300
)

print(response.choices[0].message.content)

---
# Key Modern LLMs

This section provides an overview of the most important LLMs and what makes each one notable.

## Proprietary Models

### GPT-4 / GPT-4o (OpenAI)
- **GPT-4** (March 2023) was a massive leap in capability, widely believed to use a Mixture of Experts architecture.
- **GPT-4o** (May 2024) introduced native multimodality — text, vision, and audio are processed by a single model rather than separate models piped together. This enables much faster and more natural interactions.
- GPT-4o supports a 128K token context window.

### Claude 3.5 / Claude 4 (Anthropic)
- Built using **Constitutional AI** for alignment.
- **Claude 3.5 Sonnet** (June 2024) became known for strong coding and reasoning performance.
- The Claude family offers models at different capability/cost tiers: Haiku (fast/cheap), Sonnet (balanced), Opus (most capable).
- Supports a 200K token context window with strong long-context performance.
- Supports vision (images and PDFs) alongside text.

### Gemini 1.5 / 2 (Google DeepMind)
- **Gemini 1.5 Pro** introduced a **1 million token context window** — enough to process entire codebases or hour-long videos.
- Uses a MoE architecture for efficient processing of these massive contexts.
- Natively multimodal: text, images, audio, and video.

## Open-Weight Models

### Llama 3 (Meta)
- Released in April 2024 in sizes of 8B, 70B, and 405B parameters.
- Trained on 15 trillion tokens — much more data relative to model size than previous models.
- Uses GQA, RoPE, and a 128K token context window.
- The 405B model approaches GPT-4-level performance.
- Released under a permissive license enabling commercial use.

### Mistral / Mixtral (Mistral AI)
- **Mistral 7B** — Introduced sliding window attention; punches well above its weight class.
- **Mixtral 8x7B** — One of the first widely successful open MoE models. 8 experts, 2 active per token. Total ~47B params, active ~13B.

### DeepSeek-V3 / DeepSeek-R1 (DeepSeek)
- **DeepSeek-V3** — 671B total params, 37B active. Uses innovative multi-head latent attention and auxiliary-loss-free load balancing for its MoE routing.
- **DeepSeek-R1** — A reasoning model that shows its chain-of-thought process, similar to OpenAI's o1. Notably, it was trained using large-scale RL rather than relying heavily on supervised data for reasoning.

### Qwen 2.5 (Alibaba)
- Strong multilingual performance, especially for Chinese and English.
- Available in sizes from 0.5B to 72B.
- Competitive with Llama 3 across many benchmarks.

---
# Using LLMs via APIs

The most common way to use LLMs in applications is through APIs. Here are examples using the major providers.

### OpenAI API (GPT-4)

In [ ]:
from openai import OpenAI

api_key = ''  # Add your OpenAI API key
client = OpenAI(api_key=api_key)

# Basic chat completion
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful teaching assistant for a deep learning course."},
        {"role": "user", "content": "Explain the difference between RLHF and DPO in 3 sentences."}
    ],
    temperature=0.7,  # Controls randomness (0 = deterministic, 1 = creative)
    max_tokens=200
)

print(response.choices[0].message.content)
print(f"\nTokens used - Prompt: {response.usage.prompt_tokens}, Completion: {response.usage.completion_tokens}")

### Anthropic API (Claude)

In [ ]:
!pip install anthropic

In [ ]:
import anthropic

api_key = ''  # Add your Anthropic API key
client = anthropic.Anthropic(api_key=api_key)

# Basic message
message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    system="You are a helpful teaching assistant for a deep learning course.",
    messages=[
        {"role": "user", "content": "What is Mixture of Experts and why is it important for modern LLMs?"}
    ]
)

print(message.content[0].text)
print(f"\nTokens used - Input: {message.usage.input_tokens}, Output: {message.usage.output_tokens}")

### Running Open-Weight Models Locally with Hugging Face

In [ ]:
from transformers import pipeline

# Load a small open-weight model
# For larger models (e.g., Llama 3 8B), you would need more GPU memory
generator = pipeline(
    "text-generation",
    model="gpt2",  # Replace with "meta-llama/Llama-3-8B" for a larger model (requires GPU + access)
    device_map="auto"
)

result = generator(
    "The future of artificial intelligence involves",
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

print(result[0]["generated_text"])

---
# Prompt Engineering

**Prompt engineering** is the practice of designing inputs to LLMs that elicit the best possible outputs. It is one of the most immediately practical skills when working with LLMs, because the same model can produce dramatically different results depending on how you phrase your request.

Why does phrasing matter so much? Remember that LLMs were trained to predict the next token based on patterns in their training data. The prompt you provide establishes a **context pattern** that the model continues. A vague prompt activates many possible continuation patterns; a precise prompt narrows the model to the specific type of output you want.

## Key Techniques

### Zero-Shot Prompting
The simplest approach — just ask the model to perform a task without any examples. This works well for common, well-defined tasks:

```
Classify the sentiment of this review as positive, negative, or neutral:
"The food was decent but the service was terrible."
```

The model has seen similar classification tasks in its training data, so it understands what to do. Zero-shot works best when the task is clear and unambiguous.

### Few-Shot Prompting
When the task is less obvious or you need the output in a specific format, providing a few **examples** dramatically improves results. The model uses these examples to infer the pattern you want:

```
Classify the sentiment:
"I loved this movie!" -> Positive
"Terrible waste of time." -> Negative
"It was okay." -> Neutral
"The food was decent but the service was terrible." ->
```

The model sees the pattern (text → label) and continues it. Few-shot prompting is especially useful for custom classification schemes, specific output formats, or domain-specific tasks where the model might not have strong priors.

### Chain-of-Thought (CoT) Prompting
For tasks requiring reasoning — math, logic, multi-step analysis — asking the model to **show its reasoning step by step** significantly improves accuracy. Without CoT, the model tries to jump directly to an answer, which often leads to errors on complex problems. With CoT, the intermediate steps serve as a "working scratchpad" that keeps the reasoning on track:

```
Q: If a store has 3 shelves with 8 books each, and 5 books are sold, how many remain?
A: Let me think step by step.
   - Total books: 3 × 8 = 24
   - Books sold: 5
   - Remaining: 24 - 5 = 19
   The answer is 19.
```

Research has shown that simply adding "Let's think step by step" to a prompt can improve math performance by 20–40% on some benchmarks. This works because it changes the output pattern from "question → answer" to "question → reasoning → answer," giving the model space to work through the problem.

### System Prompts
Most modern LLM APIs support a **system prompt** that sets the behavior, persona, and constraints for the model before any user interaction. The system prompt acts as persistent instructions that shape every response:

```python
messages = [
    {"role": "system", "content": "You are a data scientist. Respond with Python code and brief explanations. Always use pandas for data manipulation."},
    {"role": "user", "content": "How do I calculate the correlation between two columns in a DataFrame?"}
]
```

System prompts are powerful for establishing consistent behavior across many interactions — defining the model's role, output format, areas of expertise, and constraints.

### Structured Output
When integrating LLM outputs with downstream code, requesting specific output formats (JSON, markdown, tables) ensures the output can be reliably parsed:

```
Extract the following fields from this text and return as valid JSON only:
- name
- email
- phone

Text: "Contact John Smith at john@example.com or call 555-0123."
```

Many LLM APIs now support **structured output modes** that guarantee the response matches a specified JSON Schema, eliminating the need to parse free-form text.

### Prompt Engineering Examples

In [ ]:
from openai import OpenAI

api_key = ''  # Add your OpenAI API key
client = OpenAI(api_key=api_key)

def ask(system_prompt, user_prompt, model="gpt-4o"):
    """Helper function to make API calls."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content

# Example 1: Zero-shot classification
print("=== Zero-Shot Classification ===")
print(ask(
    "Classify the sentiment as positive, negative, or neutral. Respond with just the label.",
    "The new update completely broke my workflow and I'm extremely frustrated."
))

print("\n=== Chain-of-Thought Reasoning ===")
print(ask(
    "You are a math tutor. Show your reasoning step by step.",
    "A train leaves Station A at 9 AM traveling at 60 mph. Another train leaves Station B (180 miles away) at 10 AM traveling toward Station A at 90 mph. When do they meet?"
))

print("\n=== Structured Output (JSON) ===")
print(ask(
    "Extract information and return valid JSON only. No other text.",
    'Extract name, company, and role: "Sarah Chen is the VP of Engineering at Acme Corp."'
))

---
# Summary

Modern LLMs have evolved far beyond the original transformer architecture through innovations at every level:

| Area | Key Innovations |
|------|----------------|
| **Architecture** | RoPE, GQA, Flash Attention, MoE, Sliding Window Attention |
| **Training** | Pretraining → SFT → RLHF/DPO pipeline; Constitutional AI |
| **Scaling** | Chinchilla laws; deliberate overtraining of smaller models |
| **Fine-tuning** | LoRA, QLoRA — making customization accessible |
| **Inference** | Quantization (GPTQ, AWQ, GGUF), speculative decoding, vLLM |
| **Augmentation** | RAG for grounding outputs in external knowledge |
| **Multimodality** | Vision-language models processing text, images, audio, video |
| **Prompting** | Zero-shot, few-shot, chain-of-thought, structured output |

The field continues to evolve rapidly, with new models and techniques emerging regularly. The concepts in this notebook provide a foundation for understanding these developments as they happen.